In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import test_transforms
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir)
transformed_dataset = ImageDataset(annot_path, img_dir, test_transforms)

In [3]:
_, truth_labels = next(iter(transformed_dataset))
truth_labels.shape, truth_labels

(torch.Size([5, 5]),
 tensor([[8.0000, 0.5848, 0.7321, 0.1205, 0.3393],
         [8.0000, 0.4196, 0.8482, 0.1741, 0.2902],
         [8.0000, 0.0714, 0.8259, 0.1250, 0.3482],
         [8.0000, 0.5357, 0.6562, 0.1071, 0.2812],
         [8.0000, 0.5893, 0.5402, 0.0714, 0.0893]]))

In [4]:
bboxes = truth_labels[:, -4:]
bboxes.shape, bboxes

(torch.Size([5, 4]),
 tensor([[0.5848, 0.7321, 0.1205, 0.3393],
         [0.4196, 0.8482, 0.1741, 0.2902],
         [0.0714, 0.8259, 0.1250, 0.3482],
         [0.5357, 0.6562, 0.1071, 0.2812],
         [0.5893, 0.5402, 0.0714, 0.0893]]))

In [5]:
truth_labels[:, 0]

tensor([8., 8., 8., 8., 8.])

In [6]:
bbox_preds = torch.randn(100, 21)
bbox_preds

tensor([[ 0.0066, -0.0553, -0.7122,  ...,  0.4770,  1.6957,  0.6957],
        [-1.4537,  0.1232, -0.2455,  ...,  0.1358, -1.1601,  0.4151],
        [ 1.7849, -0.2919, -0.4380,  ..., -0.0614, -0.1195,  0.3581],
        ...,
        [-1.3677, -1.3873,  0.5240,  ..., -2.5298, -1.2480,  0.8639],
        [ 0.9754, -0.9936,  0.2652,  ...,  1.5415, -0.3483,  0.2171],
        [ 0.2600,  0.0045, -0.0494,  ..., -1.0898,  0.0555,  1.1837]])

In [7]:
class_preds = torch.randn(100, 21)

In [8]:
for idx in truth_labels[:, 0]: print(int(idx))

8
8
8
8
8


In [9]:
import torch.nn.functional as F

In [11]:
from src.utilities import compute_giou

bbox_preds = torch.randn(100, 4)

In [12]:
truth_boxes = truth_labels[..., -4:]
truth_boxes.shape

torch.Size([5, 4])

In [13]:
 truth_boxes[0], bbox_preds[0], bbox_preds[1], bbox_preds[2], bbox_preds[3], bbox_preds[4]

(tensor([0.5848, 0.7321, 0.1205, 0.3393]),
 tensor([-1.2795,  0.7263, -0.0523,  1.2996]),
 tensor([-0.4027,  0.2223,  0.3836, -0.6934]),
 tensor([ 0.4834, -0.4061,  2.2046, -0.7661]),
 tensor([-1.4724,  1.0161, -0.9675,  2.4400]),
 tensor([ 0.1695, -0.4136,  0.0271,  0.3036]))

In [14]:
preds_clone = bbox_preds.clone().detach()

# INTERSECTION COORDINATES
preds_clone[..., 0] = torch.max(preds_clone[..., 0], truth_boxes[0][0])
preds_clone[..., 1] = torch.max(preds_clone[..., 1], truth_boxes[0][1])
preds_clone[..., 2] = torch.min(preds_clone[..., 2], truth_boxes[0][2])
preds_clone[..., 3] = torch.min(preds_clone[..., 3], truth_boxes[0][3])

preds_clone.shape, preds_clone[0], preds_clone[1], preds_clone[2], preds_clone[3], preds_clone[4]

(torch.Size([100, 4]),
 tensor([ 0.5848,  0.7321, -0.0523,  0.3393]),
 tensor([ 0.5848,  0.7321,  0.1205, -0.6934]),
 tensor([ 0.5848,  0.7321,  0.1205, -0.7661]),
 tensor([ 0.5848,  1.0161, -0.9675,  0.3393]),
 tensor([0.5848, 0.7321, 0.0271, 0.3036]))

In [16]:
from src.utilities import compute_intersection_coords

preds_clone_2 = bbox_preds.clone().detach()

intersection_coords = compute_intersection_coords(preds_clone_2, truth_boxes[0])

intersection_coords

tensor([[ 0.5848,  0.7321, -0.0523,  0.3393],
        [ 0.5848,  0.7321,  0.1205, -0.6934],
        [ 0.5848,  0.7321,  0.1205, -0.7661],
        [ 0.5848,  1.0161, -0.9675,  0.3393],
        [ 0.5848,  0.7321,  0.0271,  0.3036],
        [ 0.5848,  0.7321,  0.1205, -0.1839],
        [ 1.6136,  0.7321, -1.1296, -0.2186],
        [ 0.9197,  0.7321, -0.8827, -0.8553],
        [ 0.6025,  0.7321,  0.1205, -0.0396],
        [ 0.5848,  0.7321, -0.2439, -0.9332],
        [ 1.4566,  0.7321,  0.0726,  0.3393],
        [ 0.5848,  1.1599,  0.1205,  0.3393],
        [ 1.9593,  0.7321, -0.4302, -0.6039],
        [ 0.7270,  0.7321, -1.0810,  0.3393],
        [ 0.5848,  0.7321, -0.0658, -0.6018],
        [ 1.0128,  0.7321,  0.1205, -0.7312],
        [ 0.5848,  1.5100,  0.1205,  0.3393],
        [ 0.9058,  1.4468, -0.5722,  0.3393],
        [ 1.1052,  1.0268,  0.1205,  0.0933],
        [ 0.8236,  0.7321,  0.1205,  0.3393],
        [ 0.5848,  0.7321,  0.1205,  0.3393],
        [ 0.5848,  0.7321, -1.6150

In [ ]:
w = torch.clamp(preds_clone[..., 2] - preds_clone[..., 0], min=0)
h = torch.clamp(preds_clone[..., 3] - preds_clone[..., 1], min=0)

In [ ]:
w[0], w[1], h[0], h[1], preds_clone[0], preds_clone[1]

In [ ]:
areas = w * h
areas.shape, areas[0], areas[1]